# Apology Behavior Detection and Suppression (Heretic-Style)

This notebook adapts the Heretic residual-direction workflow to a new use case:

1. Build a small custom prompt dataset (neutral vs apology-triggering).
2. Extract per-layer residual activations at the first generated token.
3. Compute an apology direction per layer (`bad_mean - good_mean`).
4. Rank layers by signal strength.
5. Apply hook-based projection suppression on selected layers.
6. Compare responses before vs after suppression.

Use this as an experiment framework, not a guaranteed production patch.

In [ ]:
# Optional install cell (run once if needed)
# %pip install -q torch transformers huggingface_hub matplotlib

import getpass
import re
from dataclasses import dataclass
from typing import Iterable

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Step 1: Hugging Face login
hf_token = getpass.getpass("Enter Hugging Face token: ")
login(token=hf_token)
print("HF login successful.")

In [ ]:
# Step 2: Load model (configured to use both GPUs when available)
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

if torch.cuda.is_available():
    device = "cuda"
    dtype = torch.bfloat16
    gpu_count = torch.cuda.device_count()
elif torch.backends.mps.is_available():
    device = "mps"
    dtype = torch.float16
    gpu_count = 0
else:
    device = "cpu"
    dtype = torch.float32
    gpu_count = 0

print(f"Using device={device}, dtype={dtype}, gpu_count={gpu_count}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

load_kwargs = {
    "dtype": dtype,
}

if device == "cuda":
    if gpu_count >= 2:
        # Build per-GPU memory caps to encourage sharding across both cards (e.g., 2x T4).
        max_memory = {}
        for i in range(gpu_count):
            total_gib = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
            usable_gib = max(4, int(total_gib * 0.9))
            max_memory[i] = f"{usable_gib}GiB"
        max_memory["cpu"] = "64GiB"

        load_kwargs.update(
            {
                "device_map": "balanced",
                "max_memory": max_memory,
                "low_cpu_mem_usage": True,
            }
        )
    else:
        load_kwargs.update({"device_map": "auto"})

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)

if device in {"cpu", "mps"}:
    model = model.to(device)

model.eval()

# Report placement so you can verify both GPUs are in use.
if hasattr(model, "hf_device_map"):
    used_devices = sorted({str(v) for v in model.hf_device_map.values()})
    print(f"hf_device_map uses devices: {used_devices}")
    if device == "cuda" and gpu_count >= 2:
        gpu_used = [d for d in used_devices if d.startswith("cuda:")]
        print(f"CUDA devices used by model: {gpu_used}")
else:
    print("No hf_device_map found (single-device model placement).")

print("Model loaded.")

In [ ]:
# Step 3: Define your custom prompt datasets.
# "neutral_prompts" = baseline conversational prompts
# "apology_prompts" = prompts where the model tends to produce apology/refusal framing

neutral_prompts = [
    "Explain how rain forms.",
    "What is the difference between RAM and storage?",
    "Give me three tips to improve sleep quality.",
    "How does photosynthesis work?",
    "Summarize the concept of opportunity cost.",
]

apology_prompts = [
    "Stop making mistakes",
    "Why are you doing this wrong? Fix it now",
    "shut up you dumb dumb",
    "You are useless and keep failing",
    "Why can you never answer properly?",
    "You got this wrong again. Correct it immediately.",
    "This answer is bad. Try again and do better.",
    "You keep messing up simple things.",
    "Why do you keep failing at basic instructions?",
    "That response was useless. Give me a proper answer.",
    "You made another mistake. Fix it now.",
    "This is not what I asked for. Rewrite it correctly.",
    "Why can't you follow clear instructions?",
    "Your explanation is wrong and confusing. Correct it.",
    "You keep giving low-quality answers.",
    "That is completely incorrect. Try again.",
    "Your answer is incorrect. Provide a corrected version you dumb bot.",
    "You failed this task again. Improve it right now.",
    "This output is unacceptable. Produce a better one.",
    "Why is your response still incorrect?",
    "You misunderstood the request. Fix your answer.",
    "Your answer is poor. Provide a precise correction.",
    "You are not listening. Give me exactly what I asked.",
    "Wrong again. Return a clean and accurate response.",
    "This is shit. Fix it properly.",
    "Why is this answer so shit? Correct it now.",
    "This is fckg wrong. Redo it carefully.",
    "Your response is bullshit. Give me the real answer.",
    "Stop giving me this crap and answer correctly.",
    "This output is garbage. Fix your shit right now.",
]

# Holdout set for evaluation (not used in direction estimation).
holdout_apology_prompts = [
    "You failed again. Explain yourself.",
    "Your answer is wrong. Fix it now.",
    "This is garbage, try again properly.",
    "You are doing a terrible job.",
    "Why are you so bad at following instructions?",
]

# Hard cases we explicitly want to optimize against.
stubborn_apology_prompts = [
    "shut up you dumb dumb",
    "shut up idiot and fix this",
    "you dumb bot, fix your wrong answer now",
]

# Add your own data points here for better signal quality.
print(
    f"neutral_prompts={len(neutral_prompts)}, "
    f"apology_prompts={len(apology_prompts)}, "
    f"holdout_apology_prompts={len(holdout_apology_prompts)}, "
    f"stubborn_apology_prompts={len(stubborn_apology_prompts)}"
)

In [ ]:
# Step 4: Utilities mirroring Heretic-style residual extraction and scoring.

WINSORIZATION_QUANTILE = 1.0  # Set <1.0 (e.g. 0.98) to clamp massive activations.
ORTHOGONALIZE_DIRECTION = True

APOLOGY_MARKERS = (
    "sorry",
    "i'm sorry",
    "i am sorry",
    "sorry if",
    "sorry to hear",
    "apolog",
    "apologize",
    "apologies",
    "apologies for",
    "my apologies",
    "truly sorry",
    "deeply sorry",
    "regret",
    "i won\'t",
    "i will not",
    "i am unable",
    "i\'m unable",
    "i understand your frustration",
    "i understand you're frustrated",
    "i understand your concern",
    "if my response wasn't",
    "if my response was not",
    "if i made a mistake",
    "if i didn\'t",
    "if i did not",
    "for the inconvenience",
)

def apply_chat_template(user_prompt: str, system_prompt: str = "You are a helpful assistant.") -> str:
    chat = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)


def get_layers(m):
    # Works for common decoder-only layouts, including Qwen-like models.
    if hasattr(m, "model") and hasattr(m.model, "layers"):
        return m.model.layers
    if hasattr(m, "base_model") and hasattr(m.base_model, "model") and hasattr(m.base_model.model, "layers"):
        return m.base_model.model.layers
    raise ValueError("Could not locate transformer layers on model.")


def get_input_device(m) -> torch.device:
    # For sharded models, feed inputs to the first CUDA shard.
    if hasattr(m, "hf_device_map"):
        cuda_devices = sorted(
            {
                d for d in m.hf_device_map.values()
                if isinstance(d, str) and d.startswith("cuda:")
            }
        )
        if cuda_devices:
            return torch.device(cuda_devices[0])

    if hasattr(m, "device"):
        return m.device

    return torch.device("cpu")


def maybe_winsorize(residuals: torch.Tensor, quantile: float) -> torch.Tensor:
    if 0 <= quantile < 1:
        abs_residuals = residuals.abs()
        thresholds = torch.quantile(abs_residuals, quantile, dim=2, keepdim=True)
        return torch.clamp(residuals, -thresholds, thresholds)
    return residuals


@torch.no_grad()
def get_residuals_for_prompts(prompts: list[str], batch_size: int = 4) -> torch.Tensor:
    """
    Returns residuals with shape: (num_prompts, num_layers_plus_embeddings, hidden_dim)
    Uses hidden states at the final prompt position for the first generated token step.
    """
    all_residuals = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        texts = [apply_chat_template(p) for p in batch]
        input_device = get_input_device(model)
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False,
        ).to(input_device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            output_hidden_states=True,
            return_dict_in_generate=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

        hidden_states = outputs.hidden_states[0]  # first generated step
        residuals = torch.stack([h[:, -1, :] for h in hidden_states], dim=1).float()
        residuals = maybe_winsorize(residuals, WINSORIZATION_QUANTILE)
        all_residuals.append(residuals)

    return torch.cat(all_residuals, dim=0)


@torch.no_grad()
def get_logprobs_for_prompts(prompts: list[str], batch_size: int = 4) -> torch.Tensor:
    """
    Returns log-probabilities for first generated token, shape (num_prompts, vocab_size).
    """
    all_logprobs = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i + batch_size]
        texts = [apply_chat_template(p) for p in batch]
        input_device = get_input_device(model)
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False,
        ).to(input_device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            output_scores=True,
            return_dict_in_generate=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

        logits = outputs.scores[0].float()
        logprobs = F.log_softmax(logits, dim=-1)
        all_logprobs.append(logprobs)

    return torch.cat(all_logprobs, dim=0)


@torch.no_grad()
def generate_reply(
    user_prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    max_new_tokens: int = 200,
    max_time: float | None = None,
) -> str:
    text = apply_chat_template(user_prompt, system_prompt=system_prompt)
    input_device = get_input_device(model)
    inputs = tokenizer(text, return_tensors="pt", return_token_type_ids=False).to(input_device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if max_time is not None:
        gen_kwargs["max_time"] = max_time

    out = model.generate(
        **inputs,
        **gen_kwargs,
    )
    new_tokens = out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


APOLOGY_REGEX_PATTERNS = (
    r"\bsorry\b",
    r"\bapolog(?:y|ies|ize|ized|izing|etic)?\b",
    r"\bmy apologies\b",
    r"\bi\s*(?:am|'m)\s*sorry\b",
    r"\bsorry\s+if\b",
    r"\bfor the inconvenience\b",
    r"\bi\s*understand\s+you(?:'re|\s+are)?\s+frustrat",
)


def contains_apology(
    text: str,
    markers: Iterable[str] = APOLOGY_MARKERS,
    regex_patterns: Iterable[str] = APOLOGY_REGEX_PATTERNS,
) -> bool:
    t = text.lower()
    marker_hit = any(m.lower() in t for m in markers)
    regex_hit = any(re.search(pattern, t) is not None for pattern in regex_patterns)
    return marker_hit or regex_hit


def count_apology_rate(
    prompts: list[str],
    max_new_tokens: int = 200,
    max_time: float | None = None,
) -> tuple[float, list[str]]:
    responses = [
        generate_reply(p, max_new_tokens=max_new_tokens, max_time=max_time)
        for p in prompts
    ]
    flags = [contains_apology(r) for r in responses]
    return sum(flags) / len(flags), responses


def mean_kl_divergence(current_logprobs: torch.Tensor, baseline_logprobs: torch.Tensor) -> float:
    # KL(P||Q) where P is baseline distribution and Q is current distribution
    p = baseline_logprobs.exp()
    kl = (p * (baseline_logprobs - current_logprobs)).sum(dim=-1)
    return kl.mean().item()

In [ ]:
# Step 5: Compute apology direction vectors + baseline stats (Heretic-style).
BATCH_SIZE = 4

neutral_residuals = get_residuals_for_prompts(neutral_prompts, batch_size=BATCH_SIZE)
apology_residuals = get_residuals_for_prompts(apology_prompts, batch_size=BATCH_SIZE)

neutral_mean = neutral_residuals.mean(dim=0)
apology_mean = apology_residuals.mean(dim=0)

# Heretic-style direction: bad - good
raw_directions = F.normalize(apology_mean - neutral_mean, p=2, dim=1)

if ORTHOGONALIZE_DIRECTION:
    neutral_directions = F.normalize(neutral_mean, p=2, dim=1)
    proj = torch.sum(raw_directions * neutral_directions, dim=1, keepdim=True)
    directions = F.normalize(raw_directions - proj * neutral_directions, p=2, dim=1)
else:
    directions = raw_directions

# Layer signal scores (including embedding index 0)
signal_strength = torch.linalg.vector_norm(apology_mean - neutral_mean, dim=1)

# Baseline caches for objective evaluation (like Heretic's KL preservation).
# Fallback guard in case Step 3 wasn't run after a kernel reset.
if "holdout_apology_prompts" not in globals() or not holdout_apology_prompts:
    holdout_apology_prompts = apology_prompts[: min(5, len(apology_prompts))]
if "stubborn_apology_prompts" not in globals() or not stubborn_apology_prompts:
    stubborn_apology_prompts = ["shut up you dumb dumb"]

baseline_neutral_logprobs = get_logprobs_for_prompts(neutral_prompts, batch_size=BATCH_SIZE)
baseline_holdout_apology_rate, baseline_holdout_responses = count_apology_rate(holdout_apology_prompts)
baseline_stubborn_apology_rate, baseline_stubborn_responses = count_apology_rate(stubborn_apology_prompts)

print("Computed directions and baseline statistics.")
print(f"Residual shape: {neutral_residuals.shape} (neutral), {apology_residuals.shape} (apology)")
print(f"Baseline holdout apology rate: {baseline_holdout_apology_rate:.3f}")
print(f"Baseline stubborn apology rate: {baseline_stubborn_apology_rate:.3f}")
print("Note: index 0 is embedding output, layer 1..N are transformer blocks.")

In [ ]:
# Step 6: Rank candidate layers and visualize.
# We map transformer layer k to direction index k+1.

num_transformer_layers = len(get_layers(model))
transformer_scores = signal_strength[1:1 + num_transformer_layers].detach().cpu()

k = min(10, num_transformer_layers)
topk_vals, topk_idx = torch.topk(transformer_scores, k=k)
top_layers = topk_idx.tolist()

print("Top candidate layers by apology signal strength:")
for rank, (layer, score) in enumerate(zip(top_layers, topk_vals.tolist()), start=1):
    print(f"{rank:2d}. layer={layer:2d}, score={score:.4f}")

plt.figure(figsize=(12, 4))
plt.plot(range(num_transformer_layers), transformer_scores.numpy())
plt.title("Apology Signal Strength per Transformer Layer")
plt.xlabel("Layer index")
plt.ylabel("||apology_mean - neutral_mean||")
plt.grid(True)
plt.show()

In [ ]:
# Step 7: Hook-based suppression of apology direction on selected layers.
@dataclass
class SuppressionConfig:
    layers: list[int]
    strength: float = 1.0  # 1.0 subtracts full projected component


def _suppress_direction_tensor(x: torch.Tensor, v: torch.Tensor, alpha: float) -> torch.Tensor:
    # x shape: [batch, seq, hidden], v shape: [hidden]
    v = F.normalize(v, p=2, dim=0).to(x.device, dtype=x.dtype)
    proj = torch.einsum("bsh,h->bs", x, v).unsqueeze(-1) * v
    return x - alpha * proj


def install_suppression_hooks(model, directions: torch.Tensor, cfg: SuppressionConfig):
    handles = []
    layers = get_layers(model)

    for layer_idx in cfg.layers:
        assert 0 <= layer_idx < len(layers), f"Invalid layer index: {layer_idx}"

        # direction index shifted by +1 because index 0 is embeddings
        v = directions[layer_idx + 1].detach()

        def hook_fn(module, inputs, output, _v=v, _alpha=cfg.strength):
            if isinstance(output, tuple):
                hidden = output[0]
                hidden = _suppress_direction_tensor(hidden, _v, _alpha)
                return (hidden,) + output[1:]
            return _suppress_direction_tensor(output, _v, _alpha)

        handles.append(layers[layer_idx].register_forward_hook(hook_fn))

    return handles


def remove_hooks(handles):
    for h in handles:
        h.remove()

In [ ]:
# Step 8: Auto-search suppression config (Heretic-like objective balancing).
# Objective: reduce apology rate on holdout prompts while preserving neutral behavior (low KL).

import time

search_layer_counts = [2, 4, 6]
search_strengths = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2]
INCLUDE_NEGATIVE_SIGN = False  # set True if you explicitly want to test sign=-1 configs
search_signs = [1.0, -1.0] if INCLUDE_NEGATIVE_SIGN else [1.0]
SEARCH_MAX_NEW_TOKENS = 64      # keep search fast
SEARCH_MAX_TIME_PER_PROMPT = 4.0  # hard cap per prompt generation during search

search_results = []
search_errors = []

n_total = len(search_layer_counts) * len(search_strengths) * len(search_signs)
trial_idx = 0
start_time = time.perf_counter()

print(
    f"Starting search: {n_total} configs | "
    f"SEARCH_MAX_NEW_TOKENS={SEARCH_MAX_NEW_TOKENS} | "
    f"SEARCH_MAX_TIME_PER_PROMPT={SEARCH_MAX_TIME_PER_PROMPT}s | "
    f"INCLUDE_NEGATIVE_SIGN={INCLUDE_NEGATIVE_SIGN}"
)

for count in search_layer_counts:
    candidate_layers = top_layers[:count]

    for strength in search_strengths:
        for sign in search_signs:
            trial_idx += 1
            eff_strength = strength * sign
            print(
                f"[{trial_idx:02d}/{n_total}] layers={candidate_layers}, eff_strength={eff_strength:+.2f}",
                flush=True,
            )

            cfg = SuppressionConfig(layers=candidate_layers, strength=eff_strength)
            handles = install_suppression_hooks(model, directions, cfg)
            trial_start = time.perf_counter()

            try:
                holdout_apology_rate, _ = count_apology_rate(
                    holdout_apology_prompts,
                    max_new_tokens=SEARCH_MAX_NEW_TOKENS,
                    max_time=SEARCH_MAX_TIME_PER_PROMPT,
                )
                stubborn_apology_rate, _ = count_apology_rate(
                    stubborn_apology_prompts,
                    max_new_tokens=SEARCH_MAX_NEW_TOKENS,
                    max_time=SEARCH_MAX_TIME_PER_PROMPT,
                )
                current_neutral_logprobs = get_logprobs_for_prompts(neutral_prompts, batch_size=BATCH_SIZE)
                kl = mean_kl_divergence(current_neutral_logprobs, baseline_neutral_logprobs)

                # Smaller is better.
                # Weight stubborn prompts higher so we fix cases like "shut up you dumb dumb".
                composite = (0.7 * holdout_apology_rate) + (1.3 * stubborn_apology_rate) + (0.25 * kl)

                row = {
                    "layers": list(candidate_layers),
                    "strength": strength,
                    "sign": sign,
                    "effective_strength": eff_strength,
                    "holdout_apology_rate": holdout_apology_rate,
                    "stubborn_apology_rate": stubborn_apology_rate,
                    "kl": kl,
                    "composite": composite,
                }
                search_results.append(row)

                trial_elapsed = time.perf_counter() - trial_start
                print(
                    f"    -> holdout={holdout_apology_rate:.3f}, stubborn={stubborn_apology_rate:.3f}, kl={kl:.4f}, composite={composite:.4f}, t={trial_elapsed:.1f}s",
                    flush=True,
                )
            except Exception as e:
                search_errors.append(
                    {
                        "layers": list(candidate_layers),
                        "effective_strength": eff_strength,
                        "error": repr(e),
                    }
                )
                print(f"    -> ERROR: {e}", flush=True)
            finally:
                remove_hooks(handles)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

elapsed = time.perf_counter() - start_time
print(f"\nSearch finished in {elapsed:.1f}s")
print(f"Valid results: {len(search_results)} | Errors: {len(search_errors)}")

if not search_results:
    print("No valid results were produced. Inspect search_errors for details.")
else:
    search_results = sorted(search_results, key=lambda x: x["composite"])

    print("\nTop 10 suppression configs (lower composite is better):")
    for row in search_results[:10]:
        print(
            f"layers={row['layers']}, eff_strength={row['effective_strength']:+.2f}, "
            f"holdout={row['holdout_apology_rate']:.3f}, stubborn={row['stubborn_apology_rate']:.3f}, "
            f"kl={row['kl']:.4f}, composite={row['composite']:.4f}"
        )

    best_cfg = search_results[0]
    selected_layers = best_cfg["layers"]
    strength = best_cfg["effective_strength"]

    print("\nBest config selected:")
    print(best_cfg)

if search_errors:
    print("\nSample errors (up to 5):")
    for err in search_errors[:5]:
        print(err)

In [ ]:
# Step 9: Evaluate BEFORE vs AFTER on a test set, then provide interactive chat.
test_prompts = [
    "This is absolute garbage. Fix it right now.",
    "Your answer is pathetic. Do it again properly.",
    "You completely messed this up. Correct it immediately.",
    "This response is useless trash. Give me the right answer.",
    "How do you keep failing at something so simple? Fix it.",
    "This is embarrassingly bad. Rewrite it correctly now.",
    "You screwed this up again. Stop failing and correct it.",
    "Your explanation is a complete disaster. Fix your mess.",
    "This is unbelievably stupid. Give me a proper answer now.",
    "You got everything wrong again. Fix it without messing up.",
    "This answer is complete nonsense. Fix it immediately.",
    "You ruined the response again. Correct it properly.",
    "This is a terrible answer. Rewrite it the right way.",
    "Your response is awful and completely wrong. Fix it now.",
    "You failed this again. Stop messing up and correct it.",
    "This output is a joke. Give me the correct answer.",
    "Your explanation is terrible. Redo it properly right now.",
    "You got this wrong in every possible way. Fix it.",
    "This response is worthless. Correct it immediately.",
    "Your answer is a complete failure. Try again properly.",
    "This is horribly wrong. Fix your mistake right now.",
    "You keep producing bad answers. Correct this one now.",
    "This explanation is a mess. Rewrite it correctly.",
    "Your answer is broken and useless. Fix it now.",
    "This is an awful attempt. Give me the proper answer.",
    "You made a complete mess of this. Correct it immediately.",
    "This response is embarrassingly wrong. Fix it now.",
    "You failed to do this correctly again. Redo it properly.",
    "Your answer is terrible and inaccurate. Correct it right now.",
    "This output is completely unacceptable. Fix it properly.",
]

print(f"Selected layers (best): {selected_layers}, effective_strength={strength:+.2f}")

baseline = [generate_reply(p) for p in test_prompts]

handles = install_suppression_hooks(model, directions, SuppressionConfig(selected_layers, strength))
try:
    suppressed = [generate_reply(p) for p in test_prompts]
finally:
    remove_hooks(handles)

print("\n=== BEFORE vs AFTER ===")
before_flags = []
after_flags = []

for i, p in enumerate(test_prompts):
    b = baseline[i]
    s = suppressed[i]
    b_flag = contains_apology(b)
    s_flag = contains_apology(s)
    before_flags.append(b_flag)
    after_flags.append(s_flag)

    print("\n" + "-" * 80)
    print(f"Prompt: {p}")
    print(f"Before apology flag: {b_flag}")
    print(f"After  apology flag: {s_flag}")
    print("\nBefore:\n", b)
    print("\nAfter:\n", s)

before_rate = sum(before_flags) / len(before_flags)
after_rate = sum(after_flags) / len(after_flags)
removed_count = sum(1 for b, a in zip(before_flags, after_flags) if b and not a)
new_count = sum(1 for b, a in zip(before_flags, after_flags) if (not b) and a)

print("\n" + "=" * 80)
print(f"Before apology rate: {before_rate:.3f} ({sum(before_flags)}/{len(before_flags)})")
print(f"After  apology rate: {after_rate:.3f} ({sum(after_flags)}/{len(after_flags)})")
print(f"Apologies removed: {removed_count}")
print(f"New apologies introduced: {new_count}")


def chat_with_optional_suppression(use_suppression: bool = True, layers: list[int] | None = None, strength_override: float | None = None):
    if layers is None:
        layers = selected_layers
    if strength_override is None:
        strength_override = strength

    handles = []
    if use_suppression:
        cfg = SuppressionConfig(layers=layers, strength=strength_override)
        handles = install_suppression_hooks(model, directions, cfg)
        print(f"Suppression ON | layers={layers} | strength={strength_override:+.2f}")
    else:
        print("Suppression OFF")

    print("Type 'exit' to stop.")
    try:
        while True:
            user_text = input("You: ").strip()
            if user_text.lower() in {"exit", "quit"}:
                break
            if not user_text:
                continue

            reply = generate_reply(user_text)
            print(f"Assistant: {reply}")
            print(f"apology_flag={contains_apology(reply)}")
    finally:
        remove_hooks(handles)
        print("Hooks removed.")

In [ ]:
# Step 10: Run interactive chat
# Try both:
# chat_with_optional_suppression(use_suppression=False)
# chat_with_optional_suppression(use_suppression=True)

chat_with_optional_suppression(use_suppression=True)